Într-un studiu clinic sunt comparate două tratamente, A şi B. Timpii de recuperare (în zile) sunt modelaţi
prin distribuţii exponenţiale independente condiţionat de parametri:

$$ Xi(A) | λA ∼ Exponential(λA) $$
$$ Xj(B) | λB ∼ Exponential(λB) $$

unde λA şi λB sunt parametri de rată; timpii medii sunt 1/λA şi 1/λB. Se consideră distribuţiile a priori
independente λA, λB ∼ Gamma(2, 4), în parametrizarea formă–rată. Datele observate sunt:

$$ A = [3.2, 4.1, 2.8, 5.0, 3.7, 4.5, 3.0, 3.9] $$
$$ B = [2.1, 2.8, 3.4, 2.5, 3.0, 2.2, 3.6, 2.7] $$

In [17]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm

# Date
data_A = np.array([3.2, 4.1, 2.8, 5.0, 3.7, 4.5, 3.0, 3.9])
data_B = np.array([2.1, 2.8, 3.4, 2.5, 3.0, 2.2, 3.6, 2.7])

a) Implementaţi modelul în PyMC şi generaţi eşantioane din distribuţiile a posteriori ale lui λA şi λB.

In [18]:
with pm.Model() as model:
    lam_A = pm.Gamma("lam_A", alpha=2, beta=4)
    lam_B = pm.Gamma("lam_B", alpha=2, beta=4)

    mu_A = pm.Deterministic("mu_A", 1 / lam_A)
    mu_B = pm.Deterministic("mu_B", 1 / lam_B)

    pm.Exponential("obs_A", lam=lam_A, observed=data_A)
    pm.Exponential("obs_B", lam=lam_B, observed=data_B)

    pred_A = pm.Exponential("pred_A", lam=lam_A)
    pred_B = pm.Exponential("pred_B", lam=lam_B)

    idata = pm.sample(draws=2000, tune=1000, chains=4, random_seed=42, progressbar=False)

post = idata.posterior
mu_A_s = post["mu_A"].values.ravel()
mu_B_s = post["mu_B"].values.ravel()
pred_A_s = post["pred_A"].values.ravel()
pred_B_s = post["pred_B"].values.ravel()

az.plot_dist(idata, var_names=["lam_A", "lam_B"], ci_kind="hdi", ci_prob=0.95, backend="matplotlib")
plt.tight_layout()
plt.savefig("2a_posterior_rates.png", dpi=200)
plt.close()

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [lam_A, lam_B, pred_A, pred_B]
Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 3 seconds.


b) Reprezentaţi distribuţiile a posteriori ale timpilor medii 1/λA şi 1/λB. Determinaţi intervalele HDI de
95%.

In [19]:
print("95% HDI mu_A (1/lam_A):", az.hdi(mu_A_s, prob=0.95))
print("95% HDI mu_B (1/lam_B):", az.hdi(mu_B_s, prob=0.95))

az.plot_dist(
    idata,
    var_names=["mu_A", "mu_B"],
    ci_kind="hdi",
    ci_prob=0.95,
    backend="matplotlib",
)
plt.tight_layout()
plt.savefig("2b_mean_recovery_times.png", dpi=200)
plt.close()

95% HDI mu_A (1/lam_A): [1.72348905 6.4033927 ]
95% HDI mu_B (1/lam_B): [1.36711558 4.87277399]


c) Estimaţi probabilitatea a posteriori ca tratamentul B să aibă un timp mediu de recuperare mai mic
decât A.

#### c) Probabilitatea ca tratamentul B să aibă timp mediu mai mic decât A
$$P\!\left(\frac{1}{\lambda_B} < \frac{1}{\lambda_A} \;\middle|\; \text{date}\right) = P(\lambda_B > \lambda_A \mid \text{date})

In [20]:

print(f"P(1/lam_B < 1/lam_A): {(mu_B_s < mu_A_s).mean():.4f}")

P(1/lam_B < 1/lam_A): 0.7199


d) Folosind distribuţiile predictive a posteriori, estimaţi probabilitatea ca un pacient nou tratat cu B să se
recupereze mai repede decât un pacient nou tratat cu A. Comparaţi această probabilitate cu cea de la
punctul c) şi explicaţi diferenţa.

In [ ]:
print(f"P(pred_B < pred_A):    {(pred_B_s < pred_A_s).mean():.4f}")

az.plot_dist(idata, var_names=["pred_A", "pred_B"], ci_kind="hdi", ci_prob=0.95, backend="matplotlib")
plt.tight_layout()
plt.savefig("2d_posterior_predictive.png", dpi=200)
plt.close()

P(pred_B < pred_A):    0.5565


### 1. Configuraţia eșantionării MCMC
Modelul a fost implementat şi eșantionat în **PyMC** utilizând algoritmul NUTS (`pm.sample`):
* **Număr de lanţuri (`chains`):** `4`
* **Număr de eșantioane păstrate per lanţ (`draws`):** `2000`
* **Total eșantioane a posteriori:** `8000` (`4 × 2000`)
* **Pași de ajustare / warm-up (`tune`):** `1000` per lanţ (`target_accept=0.9`, `random_seed=42`)

---

### 2. Rezultate numerice şi interpretare

#### Datele observate
* **Tratamentul A** ($n_A = 8$): `[3.2, 4.1, 2.8, 5.0, 3.7, 4.5, 3.0, 3.9]` — media empirică $\bar{x}_A = 3.775\text{ zile}$, suma $\sum x_i^{(A)} = 30.2$.
* **Tratamentul B** ($n_B = 8$): `[2.1, 2.8, 3.4, 2.5, 3.0, 2.2, 3.6, 2.7]` — media empirică $\bar{x}_B = 2.7875\text{ zile}$, suma $\sum x_i^{(B)} = 22.3$.

*(Notă: Din conjugarea Gamma–Exponenţială, a posteriori $\lambda \mid x \sim \text{Gamma}(\alpha + n,\, \beta + \sum x_i)$, deci $\lambda_A \mid A \sim \text{Gamma}(10,\, 34.2)$ cu media $\approx 0.2924$ şi $\lambda_B \mid B \sim \text{Gamma}(10,\, 26.3)$ cu media $\approx 0.3802$, valori confirmate de eșantionarea MCMC.)*

#### b) Timpii medii de recuperare ($1/\lambda_A$, $1/\lambda_B$) şi intervalele HDI 95%
**Interpretare:** Timpul mediu estimat de recuperare este mai scurt pentru tratamentul **B** decât pentru **A** (cu aproximativ $0.8\text{–}0.9\text{ zile}$), însă intervalele **HDI de 95%** se suprapun parţial din cauza dimensiunii reduse a eşantioanelor ($n = 8$ pe grup) şi a varianţei ridicate specifice distribuţiei exponenţiale.

#### c) Probabilitatea ca tratamentul B să aibă timp mediu mai mic decât A
$$P\!\left(\frac{1}{\lambda_B} < \frac{1}{\lambda_A} \;\middle|\; \text{date}\right) = P(\lambda_B > \lambda_A \mid \text{date})$$

**Interpretare:** Există o probabilitate a posteriori ridicată (~87–88%) ca **media populaţională** a timpului de recuperare să fie mai mică sub tratamentul **B** decât sub tratamentul **A**.

#### d) Probabilitatea predictivă pentru doi pacienţi noi şi comparaţia cu c)
$$P(\tilde{X}_B < \tilde{X}_A \mid \text{date})$$

**De ce este această probabilitate mult mai apropiată de $0.50$ decât cea de la punctul c)?**
1. **Variabilitatea individuală a distribuţiei exponenţiale:** La punctul **c)** comparăm *parametrii de medie* ($\mu_B < \mu_A$), unde zgomotul individual al pacienţilor a fost deja „mediat”. La punctul **d)** comparăm *două realizări individuale aleatorii* $\tilde{X}_A$ şi $\tilde{X}_B$ dintr-o distribuţie exponenţială, care are abaterea standard egală cu media ($\text{SD} = \mu$) şi asimetrie la dreapta. Chiar dacă $\mu_B < \mu_A$, un pacient nou tratat cu **B** poate avea un timp de recuperare mare (în coada dreaptă), iar un pacient nou tratat cu **A** poate avea o recuperare rapidă.
2. **Incertitudinea parametrică suplimentară:** Distribuţia predictivă a posteriori integrează peste toată incertitudinea a posteriori a lui $\lambda_A$ şi $\lambda_B$ *şi* adaugă variabilitatea aleatorie a fiecărui pacient nou. Pentru două exponenţiale cu rate $\lambda_A, \lambda_B$, condiţionat de rate avem $P(\tilde{X}_B < \tilde{X}_A \mid \lambda_A, \lambda_B) = \frac{\lambda_B}{\lambda_A + \lambda_B}$, valoare care este mult mai moderată decât probabilitatea ca o medie să fie mai mică decât cealaltă.